# Generative Adversarial Networks (2014)
---
[[paper]](https://arxiv.org/pdf/1406.2661)

Генеративные сети = сети, которые на выходе выдают случайно сгенрированный образ (картинка, текст, временной ряд). Образ должен быть достаточно случайным, чтобы быть уникальным, но при этом быть неотличимым от реальных данных. В идеале как можно точнее смоделировать распределение образов

Генеративные модели 

GAN (генеративные соревнующиеся сети) - подход, появившийся в 2014 году после выхода статьи GoodFellow и Bengio. Подход стал SOTA решением в области генеративного моделирования на следующие неcколько лет. Для домена Computer Vision это как минимум период с 2014 по 2018 год, пока не появились диффузионки. А для некоторых других доменов вполне используются и сейчас

Что было до: energy-based models, RNNs, Autoencoders, 

__Идея:__ есть сеть называемая "генератором", задача которой возвращать случайный сгенерированный образ. В качестве регуляризатора для нее используется другая сеть, которая называется "дискриминатором". Ей ничего не известно про генератор, но её задача распознавать плохо сгенерированные образы, предсказывая настоящий это сигнал или фейковый

Тогда решение представляется как антогонистическая игра генератора и дискриминатора. Имея real и fake примеры, дискриминатор ищет кодировку входного сигнала D(x), которая бы больше "разводила" по сторонам эти два класса и тем самым снижала ошибку предсказания. Генератор же пытается генерировать такие примеры, чтобы они попадали в те области, где дискриминатор оценивает наблюдаемые образы как реальные. Равновесное решение этой игры ($D^*$,$G^*$) соотвествует оптимальной настройке генератора, когда дискриминатор перестает отличать фейковые генерации от настоящих данных. Оптимизируется целевая функция V(D,G)

$$
\min_G \max_D \; V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}} \big[ \log D(x) \big] + \mathbb{E}_{z \sim p_z} \big[ \log (1 - D(G(z))) \big]
$$

__Интерпретация:__ для "реальных" примеров дискриминатор D максимизирует средний логарифм вероятности реальных данных $D(x)$, а для сгенерированных примеров максимизирует логарифм вероятности, что это фейковые данные 1 - D(G(z)). С другой стороны, генератор стремится  минимизировать этот штраф за фейковые данные, делая их более качественными

Архитекутура получила развитие в области CV и стала SOTA решением в период с 2015 по 2020 года

Случайный шум моделирует разнообразие $z ~ N$, сеть-генератор G() переводит его в распределение $p_{model} = G(z)$. Дискриминатор 

<img src="img/gan1.png" width=500>

__Алгоритм обучения GAN сети__<br>
Повторять 1 и 2 до сходимости:
1) <u>update дискриминатора</u>
    - генератор создает батч случайных "фейковых" данных
    - дискриминатор сэмплирует батч случайно выбранных "реальных" данных из обучающей выборки
    - оба батча объединяются в один датасет и помечаются как 0 и 1 соотвественно
    - прогоняется через дискриминатор и тот возвращает предикт вероятности реальных данных $D(x) = P(real|x)$ 
    - считаем кросс-энтропийный лосс на выходе дискриминатора<br>$
    L_D = \mathbb{E}_{x \sim p_{data}} \big[ \log D(x) \big] \;+\; 
    \mathbb{E}_{z \sim p(z)} \big[ \log (1 - D(G(z))) \big]
    $<br>на реальных данных выход дискриминатора P должен быть близок к 1.0, а на фейковых данных близок к 0.0
    - веса генератора фиксируются и мы оптимизируем только дискриминатор<br>таким образом он учится отличать реальные образы от сгенерированных<br><br>
2) <u>update генератора</u>
    - генератор создает батч случайных фейковых данных
    - считаем лосс<br>$L_G = \mathbb{E}_{z \sim p(z)} \big[ \log (1 - D(G(z))) \big]$<br>интерпретация сгенерированные данные 
    - дискриминатор замораживается и мы оптимизируем генератор<br>он учится генерировать так чтобы дискриминатор дал высокую оценку - не смог отличить от реальных данных<br><br>

### Теория

Последовательно обучая сеть, мы постепенно сводим распределение генерируемых данных $P_{model}$ к распределению реальных данных $P_{real}$. Качество дискриминатора при этом будет ухудшаться, стремясь к вероятности 1/2

<img src="img/gan2.png" width=500>

### Дискриминатор

Из постановки задачи видно, что на обоих шагах оптимизируется один и тот же функционал $V(D,G)$. Разница лишь в том, но на первом шаге зафисирован $G$, а на втором $D$. Кроме того

Функция потерь дискриминатора не что иное, как бинарная кросс-энтропия:<br>
$
L_D = - \mathbb{E}_{x \sim p_{\text{data}}} \, \log D(x) - \mathbb{E}_{z \sim p_z} \, \log \big(1 - D(G(z))\big)
$

Возможно не совсем интуитивно, почему это кросс-энтропия - там и суммы по разным распределениям идут, ещё и вход разный D(x) D(G(x)). Ниже объяснение

Напомним, что бинарная кросс-энтропия $L^{CE}$ (расстояние между $P_{pred}$ и $P_{real}$) выглядит так:<br>
$L^{CE} = - \sum_{pos} \log (P) + \sum_{neg} \log (1-P)$

Или, если занести под одну сумму<br>
$L^{CE} = - \sum \bigg( [y=\text{neg}] \cdot \log{(1-P)} + [y=\text{pos}] \cdot \log{(P)}) \bigg)$

В GAN-ах в роли двух классов выступают $\text{fake}$ и $\text{real}$:<br>
$L = - \sum \bigg( [y=\text{fake}] \cdot \log{(1-D)} + [y=\text{real}] \cdot \log{(D)}) \bigg)$

Выход дискриминатора (по своему построению) это всегда вероятность $P(real)$, а пришел сигнал из батча фейковых данных или батча настоящих не важно

### Генератор
На втором шаге оптимизации (генератора $G$) дискриминатор зафиксирован $D$, поэтому первое слагаемое можно выкинуть:

$$
L_G^{\text{minimax}} = \mathbb{E}_{z \sim p_z}\big[\log(1 - D(G(z)))\big]
$$

У такого обучения есть один минус: если нарисовать функцию поетрь, то при значениях $D=0$ (генериатор рисует плохо) производная равна -1. С учетом того, что градиент распадается на произведение более мелких градиентов, они могут получиться тченб маленькими и процесс будет сходиться неопрадванно долго

__Идея:__ поменять функцию потерь на сонаправленную, но чтобы производная была побольше

$$
L_G^{\text{non-saturating}} = - \mathbb{E}_{z \sim p_z}\big[\log D(G(z))\big]
$$

Поэтому авторы вводят так называемый non-saturarted loss. Решение то же, но сходится быстрее, так как меньше тупит на начальном этапе обучниея генератора

<img src="img/gan_trick.png" width=500>

### Связь с JS дивергенцией

У оптимизируемого функционала $V(D,G)$ есть более короткая форма записи - через одну функцию, через JSD (Jensen-Shannon дивергенцию).

Напомним, что это такое. JSD дивергенция - это симметричный вариант KL дивергенции. Принцип тот же$$
\text{JSD}(p_{\text{data}} \| p_g) =
\tfrac{1}{2}\text{KL}(p_{\text{data}} \| m) +
\tfrac{1}{2}\text{KL}(p_g \| m)
$$

Согласно постановке задачи, мы максимизируем сумму Cross-entropy loss на нашем наборе из реальных и фейковых примеров:<br>
$
p_{\text{data}}(x)\,\log D(x) + p_{fake}(x)\,\log (1 - D(x)) \rightarrow \max
$

Здесь $P_{data}$ и $P_{fake}$ это априорных вероятностей, а D(x) это правдоподобия P("real"|x)

Необходимое условие оптимальности дискриминатора $D$ выражаем через производную<br>
$
\nabla L = \frac{p_{\text{data}}(x)}{D(x)} - \frac{p_g(x)}{1 - D(x)} = 0
$

То есть для любого генератора G дискриминатор будет оптимален, когда вероятность реальных данных равнра<br>
$
D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}
$

Далее можно это учловие подставить в основное выражение:<br>
$
V(D^*, G) =
\int p_{\text{data}}(x) \log \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)} \, dx
+ \int p_g(x) \log \frac{p_g(x)}{p_{\text{data}}(x) + p_g(x)} \, dx
$

Если обозначим знаменатель за $2m(x)$, то первое слагаемое:<br>
$
\log \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}
= \log \frac{p_{\text{data}}(x)}{2 m(x)}
= \log \frac{p_{\text{data}}(x)}{m(x)} - \log 2
$

А второе:<br>
$
\log \frac{p_g(x)}{p_{\text{data}}(x) + p_g(x)}
= \log \frac{p_g(x)}{m(x)} - \log 2
$

И суммарно преобразуется в следующее вырадение:<br>
$
V(D^*, G) =
\text{KL}(p_{\text{data}} \| m) + \text{KL}(p_g \| m) - 2 \log 2
$

Jensen-Shahnnon divergence - это симметричная модификация дивергенции Кульбака-Либлера<br>
$
\text{JSD}(p_{\text{data}} \| p_g) =
\tfrac{1}{2}\text{KL}(p_{\text{data}} \| m) +
\tfrac{1}{2}\text{KL}(p_g \| m)
$

Тогда оптимизируемую функцию $V(D,G)$ можно выразить явно через JSD<br>
$
V(D^*, G) = -\log 4 + 2 \cdot \text{JSD}\!\big(p_{\text{data}} \,\|\, p_g\big)
$

## Training chanllenges and Mode Collapse
Из документации PyTorch:<br>
"Be mindful that training GANs is somewhat of an art form, as incorrect hyperparameter settings lead to mode collapse with little explanation of what went wrong..."

Некоторые хаки перечислены здесь: https://github.com/soumith/ganhacks

При обучении GAN-ов делается двухшаговая минимаксная оптимизация. Это не стандартная Loss minimiztion, у неё нет гарантий сходимости, плюс есть целый ряд негативных эффектов, собирательно называемых "training instability"

Поэтому говорят, что обучать GANы очень трудно. Это в том числе объясняет появление многочисленных модификаций в семействе GAN методов - все они призваны каким-то образом улучшить стабильиность обучения

Какие проблемы:
- нет гарантии сходимости
- малые градиенты
- Mode Collapse
- дискриминатор
- 











# DCGAN
---

[[paper]](https://arxiv.org/abs/1511.06434?utm_source=chatgpt.com)

DCGAN = Deep Convilutional GAN

Они одними из первых применили GAN подход для генерации картинок на базе CNN архитекутуры (и генератор, и дискриминатор реализуются CNN сетями) и достигли достойных результаты на небольших картнках.

Что было до этого: VAE, Energy-based models, HMMs, PixelRNN и т.д.<br>
Какие модификации были после: CycleGAN (2017), BigGAN (2018) от DeepMind, StyleGAN (2019) от Nvidia

<img src="img/dcgan.png" width=500>

Ключевые фичи модели DCGAN:
- свертки
- обучаемый пулинг (upsampling/downsampling)
- BatchNorm для борьбы с Mode Collapse
- ReLU, LeakyReLU
- FCN?



Принцип работы генратора: сеть должна научиться мапить простое распределение (шум как правило делают нормальным) в выученное распределение данных

Ответ дискриминатора: эта картинка больше похожа на то, что я видел в реальных данных, или на то, что пришло от генератора

# Conditinal GAN
---
[[paper]](https://arxiv.org/abs/1411.1784)

CGAN = conditional GAN

Two challenges:
- Gradient vanishing
- Mode collapse

# Other GAN variants
---
https://arxiv.org/pdf/1711.10337

Здесь есть неплохой обзор модификаций к типовому подходу

Подходы можно объединить:
1. смягчение функции дискриминатора
2. расширение распределений

# WGAN (2017)
---
[[paper]](https://arxiv.org/pdf/1701.07875)

WGAN = Wasserstein GAN

__Идея:__ вместо нестабильно сходящейся кросс-энтропии, будем оптимизировать расстояние Вассершейна (точнее приближенный его вариант разницу между матожиданиями). Такое расстояние лучше оптимизируется, так как градиенты всегда адекватной величины

Архитектура точно такая же, как vanilla GAN кроме следующих изменений:
- дискриминатор теперь вместо вероятности возвращает логиты, просто некий скор $f(x) in \mathrm{R}$ и называется "критиком" (Critique)<br>название выбрано так как "дискриминатор" делает выбор в пользу одной из двух альтернатив $(p, 1-p)$, а "критик" - просто даёт оценку $f(x) \rightarrow score$
- выход модели-критика делаем гладким: добавляем условие на k-Липшицовость<br>это необходимо, так как выход модели-критика теперь ничем не ограничен и может "улетать" в бесконечность 
- вместо кросс-энтропии (интеграла "несоотвествия" вероятностей) в качестве целевой функции используем <u>расстояние Вассерштейна</u><br>важно - не саму меру, а её  аппроксимацию
- Gradient Penalty
- Conditional Statement

__Wasserstein Distance__ aka __Earth Movers distance__ = сколько работы нужно совершить, чтобы "перетащить" одно распределение в другое. В отличие от кросс-энтропии такое расстояние учитывает положение двух плотностей друг относительно друга, поэтому до полной сходимости расстояния градиент всегда будет ненулевым

### Алгоритм

Параметры алгоритма:<br>
$\alpha = 0.00005$: learning rate<br>$c = 0.01$ величина клиппинга (мин/макс допустимые значения весов)<br>$m=64$ - размер батча для обучения<br>
$n_{critic} = 5$ кол-во обрабатываемых батчей для каждой итерации обучения модели-критика<br>
$w_0$ = начальная инифиализация модели критика<br>
$\theta_0$ = начальная инифиализация модели генератора

1. **while** $ \theta $ has not converged **do**  
2. &emsp;**for** $ t = 0, \ldots, n_{critic} $ **do**  
3. &emsp;&emsp;Sample $ \{x^{(i)}\}_{i=1}^m \sim \mathbb{P}_r $ a batch from the real data
4. &emsp;&emsp;Sample $ \{z^{(i)}\}_{i=1}^m \sim p(z) $ a batch of prior samples
5. &emsp;&emsp;$ g_w \leftarrow \nabla_w \Big[ \tfrac{1}{m} \sum_{i=1}^m f_w(x^{(i)}) - \tfrac{1}{m} \sum_{i=1}^m f_w(g_\theta(z^{(i)})) \Big] $  
6. &emsp;&emsp;$ w \leftarrow w + \alpha \cdot RMSProp(w, g_w) $  
7. &emsp;&emsp;$ w \leftarrow clip(w, -c, c) $  
8. &emsp;**end for**  
9. &emsp;Sample $ \{z^{(i)}\}_{i=1}^m \sim p(z) $ a batch of prior samples.  
10. &emsp;$ g_\theta \leftarrow - \nabla_\theta \tfrac{1}{m} \sum_{i=1}^m f_w(g_\theta(z^{(i)})) $  
11. &emsp;$ \theta \leftarrow \theta - \alpha \cdot RMSProp(\theta, g_\theta) $  
12. **end while**

### Теория
Важное свойство расстояния Вассерштейна - есть такой принцип __дуальности Канторовича-Рубиншетйна__, по которому расстояние Вассерштейна между двумя распределениями 

$$W(\mathbb{P}_r, \mathbb{P}_g) = \inf_{\gamma \in \Pi(\mathbb{P}_r, \mathbb{P}_g)} \mathbb{E}_{(x,y) \sim \gamma} \big[ \|x - y\| \big]$$

может быть хорошо аппроксимировано разницей $f(x)$ на матожиданиях этих распределений (с добавлением требования ограниченности таких функций), что сильно упрощает расчет

$$W(\mathbb{P}_r, \mathbb{P}_\theta) = \sup_{\|f\|_L \leq 1} \mathbb{E}_{x \sim \mathbb{P}_r}[f(x)] - \mathbb{E}_{x \sim \mathbb{P}_\theta}[f(x)]$$

*Канторович в 1930е изучал трансопртную экономику, в том числе описывал метрику Вассерштйена

Принцип требует чтобы рассматривалось не любые, а множество 1-Липшиц ограниченных функций

Из матанализа помним, что липшицовой функцией называется такая функция, что $$|f(x) - f(y)| < |x-y|$$. То есть это функции с ограниченной вариативностью. Интерпретация: для любого квадрата со стороной $\delta x$ значения функции не выходят за его границы.

1-липшицевость модели-критика реализуется примитивным способом - с помощью клиппинга <u>всех</u> весов критика $w_{i,j}$: после очередного апдейта на шаге backpropagation ставшие слишком большими (по модулю) веса обрезаются по лимитирующим значения $$w = clip(w, -c, c)$$

Форма функционала не случайна. Это хорошо изученный в статистике класс метрик, называемый [Integral Probability Metrics](https://en.wikipedia.org/wiki/Integral_probability_metric). Расстояние Вассерштейна - один из наиболее известных примеров

$$d_{\mathcal{F}}(\mathbb{P}_r, \mathbb{P}_\theta) = \sup_{f \in \mathcal{F}} \mathbb{E}_{x \sim \mathbb{P}_r}[f(x)] - \mathbb{E}_{x \sim \mathbb{P}_\theta}[f(x)]
$$

В качесвте Evaluation процедуры авторы сравнили свой подод с более классическим $DCGAN$ на задаче генерации картинок. В случае с WGAN не было примеров mode collapse

#### Преимущества подхода Wasserstein GAN
Авторы выделяют два:
- интерпретируемость функции потерь
- стабильная сходимость

##### Интерпретируемость

Под интерпретируемостью здесь понимается корреляция с визуальным качеством генерации. Чуть-чуть плохие генерации должны иметь меньшее растояние, чем совсем мусорные

Ниже пример обучения дискриминатора по GAN - JS дивергенция. Видно, что её значения ничего не говорят о качестве генерации
<img src="img/wgan_perf_1.png" width=400>

Для сравнения ниже обучение модели-критика по WGAN. Видно, что есть очевидная корреляция с качеством картинки
<img src="img/wgan2.png" width=400>

##### Стабильность обучения

Здесь стабильность = частота сходимости обучения. Утверждается что почти полностью исчезает mode collapse.

<img src="img/wgan1.png" width=400>




# WGAN-GP
---
https://arxiv.org/abs/1704.00028

# DRAGAN (2017)
---
https://arxiv.org/pdf/1705.07215

# CRAMER (2017)
---
https://arxiv.org/abs/1705.10743

# CWGAN-GP (2017)
---
https://cameronfabbri.github.io/papers/conditionalWGAN.pdf

# LSGAN
---
[[paper]](https://arxiv.org/abs/1611.04076)

LSGAN = Least suqares GAN

__Идея:__ для лучшей обощаемости заменяем вероятности на логиты (скор $o(x) \in \mathbb{R}$), а оптимизируемую функцию потреь с CE на MSE

<img src="img/lsgan.png" width=500>

# GAN-GP
---
Идея штрафуем за слишком большой градиент

# Spectral Normalization
---
При обучении ограничиваем спектральную норму матрицы весов

# ACGAN (2017)
---
[[paper]](https://arxiv.org/abs/1610.09585)

AC-GAN = Auxiliary Classifier

__Идея:__ берем CGAN, но добавляем в дискриминатор добавляем ещё одну голову - классификатор класса 

Мотивация: дискриминатор учится не только детектировать “фейк”, но и выделять дискриминативные признаки классов - это улучшает оценку генераций

<img src="img/acgan1.png" width=500>

# InfoGAN (2016)
---
[[paper]](https://arxiv.org/abs/1606.03657)

# SGAN
---
SGAN = stacked GAN

[[paper]](https://arxiv.org/abs/1612.04357)



- Cross-DomainGAN
  - CycleGAN

# BigGAN (2017)
---
__Идея:__ масштабирование GAN обучения для генерации картинок с целью улучшения качества

Архитектурно это ACGAN, но добавлено несколько фичей. В том числе shared embedding.

Фичи:
- Spectral Normalization (SN)<br>идея: нормализуем все слои собственным числом матрицы $W$<br>
Применяется ко всем слоям дискриминатора и генератора<br>
Контролирует Lipschitz-константу, предотвращает градиентный взрыв/коллапс

- Orthogonal Regularization<br>
Доп. регуляризация весов генератора: поощряет ортогональность строк весовых матриц<br>
Идея — уменьшить корреляцию фичей, улучшить разнообразие

- Skip-z (z injection)<br>
Латентный вектор 𝑧<br>
z подаётся не только в первый слой, но инжектится в несколько слоёв генератора (через конкатенацию/условную нормализацию)<br>
Улучшает использование латентного пространства

- Shared Embeddings для class conditioning<br>
В BigGAN условная генерация по классам (ImageNet → 1000 классов)
Вместо отдельных эмбеддингов → используют shared embedding: один общий embedding-вектор, скалируется под конкретный класс
Снижает параметры и улучшает обобщение

- Truncation Trick<br>
При генерации: семплы берутся не из полного Normal(0,1), а из «усечённого» распределения (отсечение крайних значений)
Итог: изображения становятся чище/реалистичнее, но менее разнообразные
Большие батчи (до 2048)
Ключевой инженерный приём: обучали на огромных батчах, что дало более стабильные градиенты
Требует много GPU (TPU v3 pods)

- Huge capacity (большие сети)<br>
Увеличили ширину/глубину слоёв генератора и дискриминатора
Показали: «чем больше сеть, тем лучше качество», если есть правильная регуляризация

- Orthogonal init + careful learning rates<br>
Правильная инициализация (ортогональная) + отдельные lr для G и D

# Tabular GANs
---

CTGAN (2019)
---
[[paper]](https://arxiv.org/pdf/1907.00503)

Успех GAN-ов заставил адаптировать к работе с табличными данными. Однако обычная GAN архитектура не очень подходит для работы с таблицами 

Проблемы табличных данных, которые ухудшают генерацию:
- переменные разных типов (числовые и категориальные)
- в отличие от картинок, часто негауссовы переменные (сложные распределения)
- мультимодальные переменные (несколько пиков в распределении)
- one-hot представления
- несбалансированные категории<br>редкие категории не сильно влияют на loss, поэтому дискриминатор не уделяет им внимание

__Mode specific normalization:__<br>Давайте будем считать все числовые переменные мультимодальными и моделировать смесью распределений. Тогда каждое значение можно закодировать двумя числами: номер смеси и z-score относительно этой смеси

__Train by sample__<br>
- выбирается категориальная переменная (случайно)
- выбирается конкретная категория (пропорционально частоте)
- строится conditional vector
- он конкатенируется со случайным шумом и подается на вход в генератор
- помимо штрафа за непохожесть отдельно штрафуем, если не 

<img src="img/ctgan.png" width=500>

Они ввели методолгию оценки:
1. likelihood
2. 

# CTABGAN (2022)
---
[[paper]](https://arxiv.org/pdf/2102.08369)

- classifier
- mixed types



<img src="img/ctabgan.png" width=1000>

# CTABGAN+
---
[[paper]](https://arxiv.org/abs/2204.00401)


CasTGAN (2023)
---
[[paper]](https://arxiv.org/abs/2307.00384)

For each variable:
  use a separate generator and separate AC (auxiliary classifier)

# Time Series GANs

TimeGAN (2019)
---
Time-series Generative Adversarial Networks

https://papers.nips.cc/paper/2019/file/c9efe5f26cd17ba6216bbe2a7d26d490-Paper.pdf

DoppleGANger
---
https://dl.acm.org/doi/pdf/10.1145/3419394.3423643